# AGAP Phase 7 — public development smoke

Use the original compatible GPU environment. Upload/extract the verification kit and original base-plus-adapter export. This notebook does not install dependencies, train, merge, serve an API or approve deployment. All cells are supplied unexecuted. Do not rerun earlier training notebook cells. No holdout or target answers are needed.


In [ ]:
from pathlib import Path
import subprocess, sys, json

# Set these to your extracted folders, including any mounted Drive path.
KIT_ROOT = Path("REPLACE_WITH_EXTRACTED_VERIFICATION_KIT")
ARTIFACT_ROOT = Path("REPLACE_WITH_ORIGINAL_BASE_PLUS_ADAPTER_FOLDER")
RESULTS = Path("/content/AGAP-phase7-results")  # new directory; never overwrite
assert KIT_ROOT.is_dir() and ARTIFACT_ROOT.is_dir(), "Set both folder paths first"
INPUTS = KIT_ROOT / "development-inputs"
INPUT_SHA = "572c401a6732f9b5325d64998c1d2aed91ecb6bef2d40faf1fa89eba35147646"
EXPORT_SHA = "377ca3440cf7429d2056d500c253d4ad9313e1309c3fc91e92d2d2b6eedc90de"


## Check the existing runtime

A nonzero preflight stops the notebook. Preserve the report and resolve the original environment separately; do not bypass version or GPU checks.


In [ ]:
p = subprocess.run([sys.executable, "-m", "models.qwen.verification", "preflight", "--out", str(KIT_ROOT / "new-preflight.json")], cwd=KIT_ROOT, check=False)
assert p.returncode == 0, "Runtime unavailable; inspect new-preflight.json"


## Execute the clean baseline, then Experiment D

This runs inference only. It preserves raw outputs and schema/source failures. Generation has token/time budgets, but model loading and a stalled GPU may still require manually interrupting the process. Never interpret an incomplete directory as a successful run.


In [ ]:
subprocess.run([sys.executable, "-m", "models.qwen.verification", "run-gpu", "--inputs", str(INPUTS), "--expected-input-sha256", INPUT_SHA, "--artifacts", str(ARTIFACT_ROOT), "--expected-export-sha256", EXPORT_SHA, "--out", str(RESULTS)], cwd=KIT_ROOT, check=True)


## Check complete canonical outputs

If a variant has invalid or truncated responses, its canonical prediction file is intentionally absent. Preserve raw files and result/failure reports for review. A pass below validates schema and citations, not semantic accuracy or authenticated execution.


In [ ]:
for variant in ("baseline", "tuned"):
    predictions = RESULTS / (variant + ".predictions.json")
    if predictions.exists():
        subprocess.run([sys.executable, "-m", "models.qwen.verification", "validate", "--inputs", str(INPUTS), "--expected-input-sha256", INPUT_SHA, "--predictions", str(predictions), "--export-manifest", str(ARTIFACT_ROOT / "model_manifest.json"), "--expected-export-sha256", EXPORT_SHA, "--variant", variant, "--out", str(RESULTS / (variant + ".validation.json"))], cwd=KIT_ROOT, check=True)
    else:
        print(variant, "has no complete canonical predictions; inspect preserved raw outputs")


## Return evidence for independent review

Preserve the entire results directory, kit checksum and original model export. Earl retains independent scoring/holdout; Aaron and Robert retain the single application adapter and registration. No model promotion, financial recommendation, 5C or goal-guidance acceptance is implied.
